# Global CDR scaling — DGSA S-curve schedules 

Paired version of `cdr-gcam-globalscaling-constApp-multiSchedule-COILED.ipynb` for dgsa. Each schedule in `synthetic-dgsa-v1.X` is scaled against its single paired geochem combination. One task per schedule runs all sites plus the lag. Shared inputs load on a worker from S3 instead of being scattered from this kernel (big scatters stall and don't reset Coiled's idle timer).

In [7]:
import os

import numpy as np
import pandas as pd
import xarray as xr
from distributed import progress

from ew_workflows.cflx_proc import scepter_rco2

In [8]:
EXP_GROUP = "gcam_dgsa_v1.2"
EXP_SET = "exe_central1p5"
SAVE_GROUP = "gcam"
idx = "001"
DUSTSP = "gbas"  # just for RCO2
SCURVE_SET = "synthetic-dgsa-v1.2"

S3 = "s3://carbonplan-carbon-removal/ew-workflows-data"
datpath = f"{S3}/scepter/postprocessed/{SAVE_GROUP}/{EXP_GROUP}/{EXP_SET}_{idx}"
SCURVE_DIR = f"{S3}/gcam/apprate_calculated/one-cell-per-region/{SCURVE_SET}"
ROCK_PARQUET = f"{SCURVE_DIR}/rock_schedules.parquet"
DSAG_ZARR = f"{S3}/gcam/apprate_calculated/one-cell-per-region/{EXP_SET}/5.0-tHaYr-appRate-FIXED/cropland+gcamRegions_time_1deg.zarr"
savedir = f"{datpath}_multiSchedule/{SCURVE_SET}"
SAVE_FN_SUM = "ds_cdr_globalScaling_constApp_SUM.zarr"

cdr_var = "cdr_adv"
RCO2_GCAM = 0.3  # [tCDR/tRock]
rco2_correction_factor = RCO2_GCAM / scepter_rco2[DUSTSP]
PERCENTILES = [0.25, 0.50, 0.75]
N_WORKERS = 40

# same lists as cdr-gcam-globalscaling-constApp*.ipynb
_cdr = ["cdr_adv", "cdr_dif", "cdr_dif_component", "cdr_resp_component", "cdr_resp_component_noAE", "cdr_SIConly",
        "co2pot_adv_tonHaYr_camg", "co2pot_adv_tonHaYr_totcat", "co2pot_tot_tonHaYr_camg", "co2pot_tot_tonHaYr_totcat"]
_dic = ["DICpot_adv_camg", "DICpot_adv_totcat"]
_frac = ["fraction_remaining_dissolved", "fraction_sld_advected", "fraction_sld_remaining", "fraction_total_dissolved"]
_src = ["noncarbsld_source_ca", "noncarbsld_source_mg", "noncarbsld_source_k", "noncarbsld_source_na"]
_catbudget = ["catbudget_gbas", "catbudget_adv", "catbudget_tflx", "catbudget_sic", "catbudget_other", "catbudget_residual"]
VARLISTS = dict(
    varlist_deploy=_cdr + _dic + _frac + ["int_dust_ton_ha_yr"] + _src + ["total_dissolution"] + _catbudget,
    varlist_region=_cdr + _catbudget,
    varlist_region_cumfrac=_frac,
    varlist_rco2_correction=_cdr + _dic + _src + _catbudget,
)
PARAMS = ["log10_C0_over_L", "k", "log10_L", "C0", "L", "t_mid", "lhs_sample_id",
          "apprate_base", "cec_factor", "dustrad", "qrun_factor"]

## Load + check

In [9]:
def load_ds(datpath):
    import xarray as xr
    return xr.open_zarr(f"{datpath}/ds_cdr.zarr").load().set_coords("apprate_base")


def load_rock(path):
    import pandas as pd
    cols = ["schedule_name", "reg_id", "region_nospaces", "time", "effective_rate_t_ha", "gcam_gross_cdr_mtCO2"]
    return dict(list(pd.read_parquet(path, columns=cols).groupby("schedule_name")))


def load_dsag(path):
    import xarray as xr
    return xr.open_zarr(path).load().drop_encoding()


manifest = pd.read_csv(f"{SCURVE_DIR}/manifest.csv", index_col="schedule_name")
ds = load_ds(datpath)
assert not ds[cdr_var].isnull().all("time").any(), "missing (site, draw) runs in ds_cdr"

rate = pd.read_parquet(ROCK_PARQUET, columns=["schedule_name", "effective_rate_t_ha"]).groupby("schedule_name").first()
paired = ds["apprate_base"].isel(site=0).sel(collapsed_inputs=manifest["collapsed_inputs"].to_numpy(dtype=str))
assert np.allclose(rate["effective_rate_t_ha"].reindex(manifest.index), paired), "schedule rates != paired draws' apprate_base"
print(f"{len(manifest)} schedules x {ds.sizes['site']} sites")

3000 schedules x 7 sites


## Cluster

In [10]:
import coiled
from dask.distributed import Client

cluster = coiled.Cluster(
    name="cdr-gcam-globalscaling-dgsa",
    n_workers=N_WORKERS,
    worker_vm_types=["m6i.large"],
    scheduler_vm_types=["r6i.xlarge"],
    region="us-west-2",
    spot_policy="on-demand",
    idle_timeout="1 hour",
    tags={"Project": "scepter"},
)
client = Client(cluster)
client

[2026-09-21 19:55:27,047][INFO    ][coiled] Fetching latest package priorities...
[2026-09-21 19:55:27,048][INFO    ][coiled.package_sync] Resolving your local /Users/tylerkukla/Github/iam-rtm-compare/uv.lock Python environment...
[2026-09-21 19:55:27,172][INFO    ][coiled.package_sync] Scanning 239 python packages...
[2026-09-21 19:55:27,358][INFO    ][coiled] Running pip check...
[2026-09-21 19:55:27,600][INFO    ][coiled] Validating environment...
[2026-09-21 19:55:28,603][INFO    ][coiled] Creating wheel for ~/Github/ew-workflows/src...
[2026-09-21 19:55:28,607][INFO    ][coiled] Creating wheel for ew-workflows...
[2026-09-21 19:55:28,609][INFO    ][coiled] Uploading coiled_local_src...
[2026-09-21 19:55:29,534][INFO    ][coiled] Uploading ew-workflows...
[2026-09-21 19:55:30,431][INFO    ][coiled] Creating software environment...
[2026-09-21 19:55:31,787][INFO    ][coiled] Creating Cluster (name: cdr-gcam-globalscaling-dgsa, https://cloud.coiled.io/clusters/2083894 ). This usually

<Client: 'tls://10.1.17.75:8786' processes=40 threads=80, memory=259.72 GiB>

## Scale each schedule (all sites + lag)

In [11]:
def _process_schedule(name, ci, ds, dfrock_dict, dsag, varlists, rco2_correction_factor, cdr_var, RCO2, percentiles):
    import xarray as xr
    from ew_workflows import cdr_fxns_postproc as cfp

    df_rock = dfrock_dict[name]
    outs = []
    for site in ds["site"].values:
        ds_rep = cfp._int_to_flux(ds.sel(site=site, collapsed_inputs=ci))
        out = cfp.gcam_postprocess_region_constApp(
            ds_rep=ds_rep, df_rock=df_rock, dsag=dsag,
            region_idx=int(df_rock.loc[df_rock["region_nospaces"] == str(site), "reg_id"].iloc[0]),
            rco2_correction_factor=rco2_correction_factor, cdr_var=cdr_var, **varlists,
        )
        # region-level only: deployment matrices for every schedule would be ~35 GB
        out = out.assign(M_D_region=out["M_D"].sum("deployment").assign_attrs(units="t/yr")).drop_dims("deployment")
        outs.append(out.expand_dims(site=[str(site)]))
    ds_out = xr.concat(outs, "site")

    lag_df, curves = cfp.compute_lag_table(ds_out, RCO2=RCO2, percentiles=percentiles)
    glob_df = cfp.compute_global_lag(curves, lag_df, percentiles=percentiles)
    ds_out = xr.merge([ds_out, xr.Dataset.from_dataframe(lag_df), xr.Dataset.from_dataframe(glob_df)])
    return ds_out.expand_dims(schedule_name=[name])


ds_f = client.submit(load_ds, datpath, pure=False)
dfrock_f = client.submit(load_rock, ROCK_PARQUET, pure=False)
dsag_f = client.submit(load_dsag, DSAG_ZARR, pure=False)

futures = client.map(_process_schedule, list(manifest.index), list(manifest["collapsed_inputs"]),
                     ds=ds_f, dfrock_dict=dfrock_f, dsag=dsag_f, varlists=VARLISTS,
                     rco2_correction_factor=rco2_correction_factor, cdr_var=cdr_var,
                     RCO2=RCO2_GCAM, percentiles=PERCENTILES)
progress(futures)
results = client.gather(futures)

In [12]:
ds_sum = xr.concat(results, "schedule_name", combine_attrs="override")
ds_sum = ds_sum.assign({
    "gcam_cdr_globe": ds_sum["gcam_cdr"].sum("site").assign_attrs(units="tCO2/yr"),
    f"{cdr_var}_globe": ds_sum[f"{cdr_var}_region"].sum("site").assign_attrs(units="t/yr"),
})
# S-curve + geochem params along schedule_name
ds_sum = ds_sum.merge(manifest[PARAMS].to_xarray()).assign(
    collapsed_inputs=("schedule_name", manifest["collapsed_inputs"].reindex(ds_sum["schedule_name"].values).to_numpy(dtype=str))
)

assert ds_sum.sizes["schedule_name"] == len(manifest) and ds_sum.sizes["site"] == ds.sizes["site"]
assert not ds_sum[f"{cdr_var}_region"].isnull().any(), "NaNs in regional CDR"
ds_sum

<xarray.Dataset> Size: 412MB
Dimensions:                                     (schedule_name: 3000, site: 7,
                                                 time: 80, metric: 15)
Coordinates:
  * schedule_name                               (schedule_name) object 24kB '...
  * site                                        (site) object 56B 'Brazil' .....
  * time                                        (time) float64 640B 0.0 ... 79.0
  * metric                                      (metric) object 120B 't_app_c...
Data variables: (12/63)
    rco2_correction_factor                      (schedule_name, site) float64 168kB ...
    M_GCAM                                      (schedule_name, site, time) float64 13MB ...
    mu                                          (schedule_name, site, time) float64 13MB ...
    M_new                                       (schedule_name, site, time) float64 13MB ...
    A_total_ERW                                 (schedule_name, site, time) float64 13MB ...
    cropland_ha_total                           (schedule_name, site) float64 168kB ...
    ...                                          ...
    lhs_sample_id                               (schedule_name) int64 24kB 0 ...
    apprate_base                                (schedule_name) float64 24kB ...
    cec_factor                                  (schedule_name) float64 24kB ...
    dustrad                                     (schedule_name) float64 24kB ...
    qrun_factor                                 (schedule_name) float64 24kB ...
    collapsed_inputs                            (schedule_name) <U125 2MB 'du...
Attributes:
    region:   Brazil

## Save

In [13]:
ds_sum.to_zarr(os.path.join(savedir, SAVE_FN_SUM), zarr_format=2)
savedir

's3://carbonplan-carbon-removal/ew-workflows-data/scepter/postprocessed/gcam/gcam_dgsa_v1.2/exe_central1p5_001_multiSchedule/synthetic-dgsa-v1.2'

In [14]:
client.close()
cluster.close()

[2026-09-21 19:59:53,414][INFO    ][coiled] Cluster 2083894 deleted successfully.


In [ ]:
# ---